# Лабораторная работа № 2. Прогнозирование качества воздуха в школьном классе

**Главный результат:** исследовательский вывод на основе реальных данных.

**Данные:** Sergio Trilles, *CO₂ concentration, temperature and humidity in primary classrooms during the Covid-19 Safety Measures in Spain*, Zenodo, 2021, DOI: [10.5281/zenodo.5062837](https://doi.org/10.5281/zenodo.5062837). Источник: [Trilles et al., Data in Brief, 2021](https://doi.org/10.1016/j.dib.2021.107489).

Измерения сделаны датчиками SCD30 в школах Испании. Показатель CO₂ помогает анализировать условия вентиляционной системы, но не доказывает качество обучения или состояние здоровья детей.

**Исследовательский вопрос:** можно ли предсказать концентрацию CO₂ **через 30 минут** точнее простого прогноза «останется как сейчас» и насколько результат устойчив во времени?

## 1. Что и зачем делаем

1. Изучаем измерения одного настоящего классного помещения и их качество.
2. Проверяем временной ряд: даты, пропуски, скачки, пики CO₂.
3. Преобразуем последние измерения в признаки (текущее значение, лаги и скользящее среднее).
4. Прогнозируем **концентрацию CO₂ через 6 пятиминутных шагов**.
5. Сравниваем **наивный прогноз**, Ridge и Random Forest по MAE/RMSE.
6. Интерпретируем результаты: когда система может помочь учителю и почему одного алгоритма недостаточно.

**Технический эффект:** демонстрация предиктивного мониторинга на IoT-данных и корректной временной валидации.  
**Педагогический эффект:** готовый исследовательский кейс по статистике, программированию, сенсорам и принятию решений об условиях обучения.

Порог **1000 ppm** далее используется **только как учебный уровень для визуализации**.

## Как оформлять отчёт в этом ноутбуке

**Перед запуском** укажите номер своего варианта в программной ячейке. **После каждого вычислительного этапа** заполните выделенное поле **«Ответ студента»**: объясните полученные числа и графики своими словами. Не удаляйте условия, таблицы результатов и программные ячейки.

| Оцениваемое задание | Баллы | Где писать ответ |
|---|---:|---|
| 1. Данные и датчик, качество измерений | **3** | После блока EDA |
| 2. Лаги и целевая переменная | **4** | После создания признаков |
| 3. Модели и временная проверка | **4** | После `TimeSeriesSplit` |
| 4. Метрики и графики | **3** | После финального теста |
| 5. Исследовательские выводы | **4** | Заключительный аналитический блок |
| 6. Воспроизводимость | **2** | Перед самопроверкой |
| **Всего** | **20** | |

**Важно:** автоматический расчёт метрик не заменяет интерпретацию. Поля ответов ниже оставлены незаполненными намеренно.


## 2. Какие поля есть в исходном CSV

| Поле | Значение | Использование в работе |
|---|---|---|
| `date_time` | время измерения | основная временная ось |
| `published_at` | время публикации измерения на сервере | проверка задержки передачи (не признак по умолчанию) |
| `sensor_id` | идентификатор датчика, например `CO2_01` | выбор аудитории |
| `co2` | концентрация CO₂, ppm | прогнозируемая величина |
| `temp` | температура, °C | дополнительный признак |
| `hum` | относительная влажность, % | дополнительный признак |
| `bat` | уровень батареи устройства, % | диагностика качества IoT-сенсора |
| `date` | календарная дата | справочное поле |
| `month` | месяц | справочное поле |
| `w_day` | день недели | справочное поле |
| `hour` | час | справочное поле |
| `mint` | минута | справочное поле |
| `sec` | секунда | справочное поле |
| `time` | строка времени (есть не во всех CSV) | справочное поле |

Не все исходные поля следует подавать в модель. Например, `published_at` отражает передачу данных на сервер, а `date_time` — время события.

## 3. Минимум формул

Прогноз через 30 минут при шаге 5 минут:


$$
\hat{C}_{t+6}=f(C_t,C_{t-1},C_{t-2},\overline{C}^{(W)}_t,T_t,H_t,\mathrm{hour}_t)
$$


Наивный ориентир: $\hat{C}_{t+6}=C_t$.

Скользящее среднее без заглядывания в будущее:

$$
\overline{C}^{(W)}_t=\frac{1}{W}\sum_{j=0}^{W-1}C_{t-j}
$$


Средняя абсолютная ошибка и среднеквадратичная ошибка:

$$
MAE=\frac1n\sum_{i=1}^n|C_i-\hat C_i|,\qquad
RMSE=\sqrt{\frac1n\sum_{i=1}^n(C_i-\hat C_i)^2}
$$


**Правило честной проверки:** модели обучаются только на прошлом. Целевая переменная из будущего недоступна при расчёте признаков. В `TimeSeriesSplit` устанавливается временной зазор между обучением и проверкой.

## 4. Настройка и загрузка реальных CSV

**Вариант от 1 до 25** выбирает школу, датчик и размер скользящего окна автоматически. Значения таблицы вариантов формируются следующей ячейкой; студенты **не меняют номер варианта**.

Официальный источник двух CSV: https://zenodo.org/records/5062837

Если автоматическая загрузка с Zenodo недоступна, откройте ссылку, скачайте **оба CSV** и загрузите их в файловую панель Colab (`/content`). Никаких синтетических данных при оценивании не используется.

### ✍️ До запуска: исследовательская гипотеза (часть задания 5 — 4 балла)

**Что предполагаете:** Ridge или Random Forest дадут меньшую ошибку, чем простое сохранение последнего значения CO₂? Почему? Ответ запишите **до просмотра результатов**; затем вернитесь к нему в заключении.

**Ответ студента:**

> _[Впишите гипотезу и одно обоснование.]_


In [ ]:
# Настройте только номер индивидуального варианта.
VARIANT = 25                 # замените на свой номер от 1 до 25
SEED = 42
H = 6                        # 6 × 5 минут = 30 минут
THRESHOLD = 1000             # учебный порог для анализа, ppm

import io
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(SEED)
assert 1 <= VARIANT <= 25
RECORD = 'https://zenodo.org/records/5062837'
FILES = ['CEIP_SantMiquel_Vilafames.csv', 'CEIP_LAlbea_ValldAbav2.csv']

# Цикл из двух школ и шести комнат; окна отличаются между блоками вариантов.
def variant_config(v):
    i = v - 1
    return {'variant': v,
            'school_file': FILES[i % 2],
            'sensor_num': 1 + ((i // 2) % 6),
            'window': [3, 6, 12][(i // 12) % 3]}

config = variant_config(VARIANT)
display(pd.DataFrame([variant_config(v) for v in range(1, 26)]))
print('Ваши настройки:', config)

In [ ]:
# Загружаем исходные файлы (из папки Colab либо с официального Zenodo).
# В некоторых выгрузках CSV встречаются нестандартные кавычки, поэтому есть резервный парсер.
def read_classroom_csv(content: bytes) -> pd.DataFrame:
    try:
        frame = pd.read_csv(io.BytesIO(content), low_memory=False, on_bad_lines='skip')
        if not {'date_time', 'sensor_id', 'co2'}.issubset(frame.columns):
            raise ValueError('Не распознаны обязательные столбцы')
    except (ValueError, pd.errors.ParserError):
        source = content.decode('utf-8-sig', errors='replace').replace('"', '')
        frame = pd.read_csv(io.StringIO(source), on_bad_lines='skip')
    frame.columns = frame.columns.str.strip()
    required = {'date_time', 'sensor_id', 'co2', 'temp', 'hum'}
    if not required.issubset(frame.columns):
        raise ValueError(f'Не найдены поля {required - set(frame.columns)}. Фактически: {list(frame.columns)}')
    return frame


def load_official(name):
    local = Path(name)
    if not local.is_file() and Path('/content', name).is_file():
        local = Path('/content', name)
    if local.is_file():
        data = local.read_bytes()
        source = f'локальный файл {local}'
    else:
        urls = [f'https://zenodo.org/records/5062837/files/{name}?download=1',
                f'https://zenodo.org/api/records/5062837/files/{name}/content']
        data = None
        details = []
        for url in urls:
            try:
                resp = requests.get(url, timeout=50)
                resp.raise_for_status()
                data = resp.content
                source = url
                break
            except requests.RequestException as exc:
                details.append(type(exc).__name__)
        if data is None:
            raise RuntimeError('Не удалось загрузить данные автоматически (' + ', '.join(details) + '). '
                               f'Скачайте {name} на {RECORD} и загрузите в Colab.')
    frame = read_classroom_csv(data)
    frame['sensor_id'] = frame['sensor_id'].astype(str).str.strip().str.strip('"')
    frame['date_time'] = pd.to_datetime(frame['date_time'].astype(str).str.strip(), errors='coerce')
    for col in ['co2', 'temp', 'hum', 'bat']:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors='coerce')
    print(f'{name}: {len(frame):,} строк; источник: {source}')
    return frame

raw = load_official(config['school_file'])
print('Имеющиеся идентификаторы:', sorted(raw['sensor_id'].dropna().unique()))

### ✍️ Паспорт исследования (задания 1 и 6)

После загрузки CSV перепишите фактически показанные программой параметры (не догадки).

| Параметр | Заполняет студент |
|---|---|
| Номер варианта | … |
| Школа и имя CSV | … |
| `sensor_id` выбранного датчика | … |
| Период наблюдений | … |
| Источник данных (ссылка) | … |

**Краткое пояснение:**

> _[Почему именно эти данные соответствуют вашему варианту?]_


## 5. EDA анализ: качество данных

В исходных измерениях возможны пропуски, дубли, неполадки связи и физически неправдоподобные значения. В отличие от идеального учебного массива эти ситуации **нужно описать**. Нельзя заменять будущие значения CO₂ для расчёта целевой переменной.

**Исследовательская запись № 1:** после таблицы сформулируйте два наблюдения: сколько измерений доступно и встречались ли подозрительные значения.

In [ ]:
quality = pd.DataFrame({
    'Показатель': ['Строки CSV', 'Уникальные датчики', 'Некорректные даты',
                   'Дубликаты (датчик, время)', 'Пропуски CO₂',
                   'CO₂ вне диапазона 250…10000', 'Температура вне −20…60 °C'],
    'Значение': [len(raw), raw['sensor_id'].nunique(), raw['date_time'].isna().sum(),
                raw.duplicated(['sensor_id', 'date_time']).sum(), raw['co2'].isna().sum(),
                (~raw['co2'].between(250,10000) & raw['co2'].notna()).sum(),
                (~raw['temp'].between(-20,60) & raw['temp'].notna()).sum()]
})
display(quality)

# Выбор датчика из доступных идентификаторов, упорядоченных по номеру.
ids = sorted(raw['sensor_id'].dropna().unique())
if len(ids) < config['sensor_num']:
    raise ValueError('В исходном CSV меньше датчиков, чем предполагает конфигурация. '
                     f'Доступные: {ids}; нужен номер {config["sensor_num"]}')
SENSOR = ids[config['sensor_num']-1]
print('Исследуемый датчик:', SENSOR)

one = raw.loc[raw['sensor_id'] == SENSOR, ['date_time','co2','temp','hum']].copy()
one = one.dropna(subset=['date_time'])
one.loc[~one['co2'].between(250,10000), 'co2'] = np.nan
one.loc[~one['temp'].between(-20,60), 'temp'] = np.nan
one.loc[~one['hum'].between(0,100), 'hum'] = np.nan

# Наблюдения попадают в пятиминутные интервалы, учитываем дубликаты по среднему.
one['time_bin'] = one['date_time'].dt.floor('5min')
series = one.groupby('time_bin')[['co2','temp','hum']].mean().sort_index().asfreq('5min')
print('5-минутных отметок:', len(series),
      'из них без измерения CO₂:', series['co2'].isna().sum(),
      f'({series["co2"].isna().mean():.1%})')
print('Период:', series.index.min(), '—', series.index.max())

ax = series['co2'].plot(figsize=(12,3.5), lw=0.8, title=f'CO₂ во времени · {SENSOR}')
ax.set(xlabel='Дата', ylabel='CO₂, ppm')
plt.tight_layout(); plt.show()

### ✍️ Задание 1. Данные и сенсор — **3 балла**

**Оценивание:** реальный исходный CSV, выбранный датчик, анализ пропусков, диапазонов значений и графика ряда.

**Ответ студента (после запуска кода EDA):**

- Число исходных строк и число точек выбранного датчика: **…**
- Доля пропущенных пятиминутных измерений CO₂: **… %**
- Минимум, максимум или характерные значения CO₂ (ppm): **…**
- Что видно по графику? Есть ли резкие изменения, пропуски, подозрительные измерения?

> _[Впишите 2–4 предложения с конкретными значениями из таблицы и графика.]_


## 6. Признаки и целевая переменная

- Вход: значения CO₂, температуры и влажности, известные **сейчас**; последние 1–2 измерения и скользящая статистика.
- Выход: **фактически измеренный CO₂ через 30 минут**, а не заполненное или вымышленное значение.
- При единичном пропуске признака разрешено взять последнее наблюдение (`ffill(limit=1)`); большой разрыв не заполняем. Сам целевой CO₂ никогда не восстанавливаем по соседним точкам.
- Параметр `window` вашего варианта влияет на число отсчётов в скользящем среднем.

In [ ]:
W = config['window']

# co2_raw — только настоящие наблюдения: используется для правильного ответа в будущем.
co2_raw = series['co2']
# features_current — информация доступна к моменту прогноза.
features_current = series[['co2','temp','hum']].ffill(limit=1)
F = pd.DataFrame(index=series.index)
F['co2_now']  = features_current['co2']
F['co2_lag1'] = features_current['co2'].shift(1)
F['co2_lag2'] = features_current['co2'].shift(2)
F['co2_mean'] = features_current['co2'].rolling(W, min_periods=W).mean()
F['co2_std']  = features_current['co2'].rolling(W, min_periods=W).std().fillna(0)
F['temp_now'] = features_current['temp']
F['hum_now']  = features_current['hum']
F['hour_sin'] = np.sin(2*np.pi*(series.index.hour + series.index.minute/60)/24)
F['hour_cos'] = np.cos(2*np.pi*(series.index.hour + series.index.minute/60)/24)

# ВАЖНО: цель сдвигается НА ШЕСТЬ РЕАЛЬНЫХ ВРЕМЕННЫХ ИНТЕРВАЛОВ (5 минут каждый).
F['target_30min'] = co2_raw.shift(-H)
F = F.replace([np.inf,-np.inf], np.nan).dropna()
FEATURES = [col for col in F.columns if col != 'target_30min']
print('Число пригодных примеров:', len(F), 'из', len(series))
if len(F) < 300:
    raise ValueError('Слишком мало наблюдений: проверьте исходный CSV, даты и датчик.')
display(F.head(5).round(2))

### ✍️ Задание 2. Лаги и целевая переменная — **4 балла**

**Оценивание:** признаки используют только сведения из прошлого; целевое значение относится к моменту через 30 минут.

**Ответ студента (после создания `F`):**

- Сколько обучающих примеров удалось сформировать: **…**
- Ширина скользящего окна вашего варианта (`W`): **…** измерений, то есть примерно **…** минут.
- Объясните своими словами три признака: `co2_lag1`, `co2_mean`, `hour_sin` / `hour_cos`.

> _[Почему `target_30min` нельзя подать в модель как входной признак? Как код не допускает использования будущего? Впишите 2–4 предложения.]_


## 7. Проверка временных интервалов: только прошлое → будущее

Последние 20% подготовленных записей — **итоговый тест**, он не участвует в выборе модели. В оставшихся 80% используем `TimeSeriesSplit(n_splits=3, gap=6)`: временной зазор защищает от пересечения горизонта прогноза с проверочным периодом.

Сравниваем:
- **наивный прогноз**: через полчаса CO₂ такой же, как сейчас;
- **Ridge**: линейная регрессия с регуляризацией;
- **Random Forest**: ансамбль деревьев для нелинейных зависимостей.

*Это не соревнование ради самой высокой Accuracy: задача регрессионная; анализируем MAE и RMSE в ppm.*

In [ ]:
CUT = int(0.8 * len(F))
TRAIN = F.iloc[:CUT-H].copy()  # gap до теста: обучающие метки не попадают в тестовый период
TEST = F.iloc[CUT:].copy()
Xtr, ytr = TRAIN[FEATURES], TRAIN['target_30min']
Xte, yte = TEST[FEATURES], TEST['target_30min']

models = {
    'Ridge': lambda: make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
    'Random Forest': lambda: RandomForestRegressor(n_estimators=70, max_depth=12,
                                                    min_samples_leaf=3, random_state=SEED,
                                                    n_jobs=-1)
}

def metrics(y, pred):
    return {'MAE, ppm': mean_absolute_error(y,pred),
            'RMSE, ppm': np.sqrt(mean_squared_error(y,pred))}

# Временная кросс-валидация только на train, не на финальном test.
cv = TimeSeriesSplit(n_splits=3, gap=H)
cv_rows = []
for fold, (tr_i, va_i) in enumerate(cv.split(Xtr), start=1):
    va_y = ytr.iloc[va_i]
    base_pred = Xtr.iloc[va_i]['co2_now'].values
    cv_rows.append({'fold':fold, 'model':'Наивный прогноз', **metrics(va_y, base_pred)})
    for title, new_model in models.items():
        model = new_model()
        model.fit(Xtr.iloc[tr_i], ytr.iloc[tr_i])
        pred = model.predict(Xtr.iloc[va_i])
        cv_rows.append({'fold':fold, 'model':title, **metrics(va_y, pred)})

cv_df = pd.DataFrame(cv_rows)
display(cv_df.round(2))
summary = cv_df.groupby('model')[['MAE, ppm','RMSE, ppm']].agg(['mean','std']).round(2)
display(summary)

# Для чистоты исследования выбираем по MAE на CV, затем обучаем выбранную модель
# на всей допустимой истории, не трогая TEST до финальной оценки.
winner = cv_df.groupby('model')['MAE, ppm'].mean().idxmin()
print('Лидер по CV-MAE:',winner)

### ✍️ Задание 3. Модели и временная проверка — **4 балла**

**Оценивание:** наивный прогноз, Ridge и Random Forest; временная кросс-валидация и отдельная проверочная часть.

**Ответ студента (после выполнения `TimeSeriesSplit`):**

| Модель | Средняя MAE на фолдах, ppm | Что означает результат |
|---|---:|---|
| Наивный прогноз | … | … |
| Ridge | … | … |
| Random Forest | … | … |

- Модель с наименьшей **средней** MAE по временным фолдам: **…**
- Отличается ли качество между фолдами: **…**

> _[Почему записи нельзя перемешивать случайно, зачем нужен `gap=6`, и какую роль `Pipeline` играет для Ridge? Впишите 2–4 предложения.]_


## 8. Итоговый тест и аналитические графики

Оценка делается на одном последнем периоде. Простой алгоритм может оказаться лучше сложного — это **корректный исследовательский результат**, а не ошибка задания.

In [ ]:
final_models = {}
predictions = {'Наивный прогноз': Xte['co2_now'].to_numpy()}
for title, new_model in models.items():
    fitted = new_model().fit(Xtr,ytr)
    final_models[title] = fitted
    predictions[title] = fitted.predict(Xte)

rows=[]
for title, preds in predictions.items():
    rows.append({'model':title, **metrics(yte,preds)})
results=pd.DataFrame(rows).sort_values('MAE, ppm')
display(results.round(2))

# Рисунок факта и прогнозов на нескольких последовательных учебных днях.
show_n = min(576, len(TEST))  # 48 часов 5-минутных интервалов
fig,ax=plt.subplots(figsize=(13,4))
ax.plot(TEST.index[:show_n],yte.values[:show_n],label='Измеренное CO₂', lw=1.7)
for title,pred in predictions.items():
    ax.plot(TEST.index[:show_n],pred[:show_n],label=title, lw=1,alpha=0.8)
ax.axhline(THRESHOLD,color='firebrick',lw=1,ls='--',label='Учебный порог 1000 ppm')
ax.set(xlabel='Дата', ylabel='CO₂, ppm', title=f'Тест: факт и прогноз через 30 минут · {SENSOR}')
ax.legend(ncol=3, fontsize=9); fig.autofmt_xdate(); plt.tight_layout();plt.show()

fig,axs=plt.subplots(1,2,figsize=(12,3.5))
for metric,ax in zip(['MAE, ppm','RMSE, ppm'],axs):
    ax.bar(results['model'],results[metric]); ax.set(title=metric, ylabel='ppm')
    ax.tick_params(axis='x',rotation=18)
plt.tight_layout(); plt.show()

# Проверяем ошибки именно на высоких концентрациях: это педагогически важная часть.
peak_mask=yte.to_numpy()>=THRESHOLD
print('Реальных измерений CO₂ ≥ 1000 ppm на тесте:', int(peak_mask.sum()))
if peak_mask.sum()>=5:
    peak_rows=[]
    for title,pred in predictions.items():
        peak_rows.append({'Модель':title,
                          'MAE при CO₂ ≥ 1000, ppm':mean_absolute_error(yte.to_numpy()[peak_mask],pred[peak_mask])})
    display(pd.DataFrame(peak_rows).round(2))
else:
    print('Для этой аудитории высоких значений слишком мало: не делайте вывод о надёжности предупреждения.')

### ✍️ Задание 4. Метрики и графики — **3 балла**

**Оценивание:** фактические MAE/RMSE на финальном тесте, сравнение с базовым прогнозом, содержательное чтение графиков.

**Ответ студента (после финальной оценки):**

| Модель | MAE, ppm | RMSE, ppm |
|---|---:|---:|
| Наивный прогноз | … | … |
| Ridge | … | … |
| Random Forest | … | … |

- Наименьшая MAE на итоговом тесте: **…** у модели **…**
- Изменение MAE по сравнению с наивным прогнозом: **… ppm** (лучше / хуже).

> _[Что показывает график «факт — прогноз»? Где возникают самые заметные промахи? Почему RMSE может быть больше MAE? Впишите 2–4 предложения.]_


## 9. ✍️ Задание 5. Исследовательское заключение — **4 балла**

**Оценивание:** собственная гипотеза, выводы с числовыми аргументами, техническая и педагогическая значимость, ограничения исследования. Отвечайте **по данным своего варианта**. Ориентир — 8–12 содержательных предложений в сумме.

**1. Гипотеза.** Вернитесь к предположению, записанному **до запуска**. Подтвердилось ли оно?

> _[Ответ студента.]_

**2. Качество исходных данных.** Каких измерений не хватает, каков диапазон CO₂ и чем он отличается от температуры и влажности? Почему могли возникнуть пропуски?

> _[Ответ студента с числами.]_

**3. Сравнение методов.** Какая модель лучше на финальном тесте по MAE и RMSE? Насколько она лучше или хуже простого прогноза, в ppm?

> _[Ответ студента с числами.]_

**4. Устойчивость.** Насколько различаются ошибки между временными фолдами? Можно ли без новой проверки переносить модель на другую школу?

> _[Ответ студента.]_

**5. Технический эффект.** Как выглядел бы полезный учителю интерфейс: текущее CO₂, тренд, прогноз, предупреждение? Какие данные нужны для достоверного предупреждения?

> _[Ответ студента.]_

**6. Педагогический эффект.** Предложите одну исследовательскую мини-задачу для школьников на основе полученного графика. Какие компетенции IoT и анализа данных она формирует?

> _[Ответ студента.]_

**7. Ограничения.** Почему CO₂ нельзя использовать для оценки знаний, дисциплины или здоровья конкретного ребёнка? Почему прогноз не доказывает улучшения качества воздуха?

> _[Ответ студента.]_

**Общий вывод (1–2 предложения):**

> _[Итоговый исследовательский вывод с опорой на измеренные значения.]_


## 10. ✍️ Задание 6. Воспроизводимость — **2 балла**

**Оценивание:** выбран правильный вариант, использован исходный датасет, последовательный запуск ноутбука воспроизводит таблицы и графики.

**Ответ студента:**

- Мой номер варианта: **…**; датчик: **…**
- Источник CSV (Zenodo или загруженная оттуда локальная копия): **…**
- После команды **Runtime → Restart and run all** все ячейки выполнились: **да / нет**. Если нет — поясните причину: **…**
- Итоговый файл `.ipynb` сохранён вместе с вычисленными таблицами и графиками: **да / нет**.

> _[Одной фразой укажите, какие параметры и условия необходимо сохранить, чтобы другой преподаватель мог повторить исследование.]_


## 11. Самопроверка перед сдачей

- [ ] Работал(а) только с исходным CSV Zenodo; указан вариант и датчик.
- [ ] Построены графики ряда, факт/прогноз и таблицы MAE/RMSE.
- [ ] Тестовая часть расположена **позже** обучающей; использован `TimeSeriesSplit` с зазором.
- [ ] Есть сравнение с наивным прогнозом.
- [ ] На все 7 исследовательских вопросов даны ответы с числами и ограничениями.
- [ ] Ноутбук выполняется через `Runtime → Restart and run all` (при доступных данных).

**Сдача:** один `.ipynb` с сохранёнными результатами и исследовательским заключением. Максимальная оценка — **20 баллов**.